# Proyek-1: Fitur Tambahan untuk Klasifikasi Buah Segar dan Busuk (MobileNetV2)

Notebook ini **melanjutkan Tugas-1** (model MobileNetV2 + Grad-CAM + confidence threshold) dengan **dua fitur baru**, tanpa melatih ulang model:

| Fitur baru | Isi | Yang diukur |
|---|---|---|
| **3. Kompresi model ke TensorFlow Lite** | Konversi ke TFLite float32, float16, dan int8 (dynamic range quantization) | Ukuran file, akurasi test, kecepatan inferensi |
| **4. Test-Time Augmentation (TTA)** | Prediksi dirata-rata dari beberapa versi gambar (asli, flip horizontal, flip vertikal, rotasi 180°) | Akurasi pada data test bersih dan data test yang diberi gangguan, serta penolakan input di luar domain |

**Cara menjalankan di Kaggle**
1. *Add Input* → dataset **Fruits fresh and rotten for classification**.
2. *Add Input* → **Your Work → Notebooks** → pilih notebook Tugas-1 (`notebookd31b9accff`, versi yang sukses). Notebook ini otomatis mencari file `fruit_mobilenetv2.keras` dan `config.json` dari output-nya.
3. *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
4. **Save Version → Save & Run All**.

Hasil disimpan di `/kaggle/working/proyek1/` dan dikemas menjadi `hasil_proyek1.zip`.

## 0. Setup

In [ ]:
import os, io, json, glob, time, zipfile, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import sklearn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_auc_score)
import warnings
import tensorflow as tf
import keras

warnings.filterwarnings('ignore')
INPUT_ROOT = os.environ.get('INPUT_ROOT', '/kaggle/input')
OUT_DIR    = os.path.join(os.environ.get('OUT_DIR', '/kaggle/working'), 'proyek1')
FIG_DIR, RES_DIR, MODEL_DIR = (os.path.join(OUT_DIR, d) for d in ('figures', 'results', 'model'))
for d in (FIG_DIR, RES_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
N_LATENCY = int(os.environ.get('N_LATENCY', 100))   # jumlah gambar untuk mengukur kecepatan
rng = np.random.default_rng(SEED)
sns.set_theme(style='whitegrid')
RESULTS = {}

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print('Gambar disimpan:', path)

print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Memuat model Tugas-1 dan data test

In [ ]:
def find_one(pattern):
    hits = sorted(glob.glob(os.path.join(INPUT_ROOT, '**', pattern), recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(f'{pattern} tidak ditemukan di {INPUT_ROOT}. Tambahkan output notebook Tugas-1 lewat Add Input.')
    return hits[0]

MODEL_PATH = find_one('fruit_mobilenetv2.keras')
CONFIG_PATH = os.path.join(os.path.dirname(MODEL_PATH), 'config.json')
if not os.path.exists(CONFIG_PATH):
    CONFIG_PATH = find_one('config.json')
with open(CONFIG_PATH) as f:
    CFG = json.load(f)

IMG_SIZE = CFG['img_size']
CLASSES = CFG['classes']
NAMES = CFG['display_names']
THRESHOLD = round(float(CFG['confidence_threshold']), 2)
model = keras.models.load_model(MODEL_PATH)
print('Model     :', MODEL_PATH)
print('Kelas     :', CLASSES)
print('Threshold :', THRESHOLD)

hits = glob.glob(os.path.join(INPUT_ROOT, '**', 'test', 'freshapples'), recursive=True)
DATA_ROOT = sorted({os.path.dirname(os.path.dirname(h)) for h in hits}, key=len)[0]
rows = [(os.path.join(DATA_ROOT, 'test', c, f), i) for i, c in enumerate(CLASSES)
        for f in sorted(os.listdir(os.path.join(DATA_ROOT, 'test', c)))]
test_df = pd.DataFrame(rows, columns=['path', 'y'])
print('DATA_ROOT :', DATA_ROOT)
print('Jumlah citra test:', len(test_df))

In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

t0 = time.time()
X_test = np.stack([load_image(p).numpy() for p in test_df['path']])
y_test = test_df['y'].values
print('X_test:', X_test.shape, X_test.dtype, f'({time.time()-t0:.1f} detik)')

def summarize(y, probs):
    pred = probs.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    return {'accuracy': float(accuracy_score(y, pred)), 'precision_macro': float(p),
            'recall_macro': float(r), 'f1_macro': float(f), 'errors': int((pred != y).sum())}

p_base = model.predict(X_test.astype('float32'), batch_size=BATCH_SIZE, verbose=0)
RESULTS['baseline_keras'] = summarize(y_test, p_base)
print('Model Tugas-1 pada data test:', RESULTS['baseline_keras'])

## 2. Fitur baru 1: Kompresi model ke TensorFlow Lite

**Apa:** model Keras dikonversi ke format **TensorFlow Lite**, format ringan untuk perangkat mobile, edge, dan server berspesifikasi rendah. Ada tiga varian:
- **float32**: konversi biasa tanpa kuantisasi
- **float16**: bobot disimpan sebagai float 16-bit
- **int8 (dynamic range)**: bobot dikuantisasi ke 8-bit

**Mengapa dibutuhkan:** file `.keras` berukuran sekitar 24 MB dan membutuhkan TensorFlow penuh. Model yang lebih kecil lebih cepat diunduh dan dimuat, hemat memori di server gratis (Streamlit Community Cloud, sekitar 1 GB RAM), dan bisa dipakai di aplikasi Android.

**Bukti dampak:** ukuran file, akurasi pada seluruh data test, kesamaan prediksi dengan model asli, dan waktu inferensi per gambar.

In [ ]:
def convert(kind):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if kind in ('float16', 'int8'):
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if kind == 'float16':
        conv.target_spec.supported_types = [tf.float16]
    return conv.convert()

TFLITE_PATHS = {}
for kind in ['float32', 'float16', 'int8']:
    path = os.path.join(MODEL_DIR, f'fruit_mobilenetv2_{kind}.tflite')
    with open(path, 'wb') as f:
        f.write(convert(kind))
    TFLITE_PATHS[kind] = path
    print(f'{kind:8s}: {os.path.getsize(path)/1e6:.2f} MB')

In [ ]:
class TFLiteModel:
    def __init__(self, path, threads=4):
        self.it = tf.lite.Interpreter(model_path=path, num_threads=threads)
        self.inp = self.it.get_input_details()[0]['index']
        self.out = self.it.get_output_details()[0]['index']
        self.it.resize_tensor_input(self.inp, [1, IMG_SIZE, IMG_SIZE, 3])
        self.it.allocate_tensors()
    def predict_one(self, x):
        self.it.set_tensor(self.inp, x[None].astype('float32'))
        self.it.invoke()
        return self.it.get_tensor(self.out)[0]
    def predict(self, X):
        return np.stack([self.predict_one(x) for x in X])

def keras_one(x):
    return model(x[None].astype('float32'), training=False).numpy()[0]

def latency_ms(fn, X, n=N_LATENCY):
    for x in X[:5]:
        fn(x)                                   # pemanasan
    t0 = time.perf_counter()
    for x in X[:n]:
        fn(x)
    return (time.perf_counter() - t0) / n * 1000

lat_idx = rng.choice(len(X_test), size=min(N_LATENCY, len(X_test)), replace=False)
X_lat = X_test[lat_idx]

rows = []
size_keras = os.path.getsize(MODEL_PATH) / 1e6
rows.append({'Model': 'Keras (.keras, Tugas-1)', 'Ukuran (MB)': size_keras,
             **{k: v for k, v in RESULTS['baseline_keras'].items()},
             'Prediksi sama dgn Keras': 1.0, 'Latensi (ms/gambar)': latency_ms(keras_one, X_lat)})
P_TFLITE = {}
for kind, path in TFLITE_PATHS.items():
    tm = TFLiteModel(path)
    t0 = time.time()
    probs = tm.predict(X_test)
    P_TFLITE[kind] = probs
    s = summarize(y_test, probs)
    rows.append({'Model': f'TFLite {kind}', 'Ukuran (MB)': os.path.getsize(path) / 1e6, **s,
                 'Prediksi sama dgn Keras': float((probs.argmax(1) == p_base.argmax(1)).mean()),
                 'Latensi (ms/gambar)': latency_ms(tm.predict_one, X_lat)})
    print(f'{kind}: selesai {len(X_test)} gambar dalam {time.time()-t0:.1f} detik')

tfl = pd.DataFrame(rows).set_index('Model')
tfl['Pengecilan ukuran'] = size_keras / tfl['Ukuran (MB)']
pd.set_option('display.width', 200)
print(tfl.round(4).to_string())
tfl.to_csv(os.path.join(RES_DIR, 'perbandingan_tflite.csv'))
RESULTS['tflite'] = tfl.reset_index().to_dict(orient='records')
print('\nCatatan: latensi Keras diukur di', 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU',
      '; TFLite selalu berjalan di CPU.')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
labels = tfl.index.str.replace(' (.keras, Tugas-1)', '', regex=False)
colors = ['#8C8C8C', '#4C72B0', '#55A868', '#DD8452']
ax[0].bar(labels, tfl['Ukuran (MB)'], color=colors)
ax[0].set_title('Ukuran file model (MB)')
for i, v in enumerate(tfl['Ukuran (MB)']):
    ax[0].text(i, v, f'{v:.1f}', ha='center', va='bottom')
ax[1].bar(labels, tfl['accuracy'] * 100, color=colors)
ax[1].set_ylim(max(0, tfl['accuracy'].min() * 100 - 1), 100)
ax[1].set_title('Akurasi data test (%)')
for i, v in enumerate(tfl['accuracy'] * 100):
    ax[1].text(i, v, f'{v:.2f}', ha='center', va='bottom')
ax[2].bar(labels, tfl['Latensi (ms/gambar)'], color=colors)
ax[2].set_title('Waktu inferensi per gambar (ms)')
for i, v in enumerate(tfl['Latensi (ms/gambar)']):
    ax[2].text(i, v, f'{v:.1f}', ha='center', va='bottom')
for a in ax:
    a.tick_params(axis='x', rotation=20)
plt.tight_layout(); savefig('p1_01_perbandingan_tflite.png'); plt.show()

## 3. Fitur baru 2: Test-Time Augmentation (TTA)

**Apa:** saat prediksi, satu gambar dibuat menjadi beberapa versi, lalu probabilitasnya **dirata-rata**. Versi yang dipakai: asli, flip horizontal, flip vertikal, dan rotasi 180°. Semua versi ini masih sesuai dengan augmentasi yang dipakai saat training (RandomFlip horizontal & vertikal).

**Mengapa dibutuhkan:** prediksi dari satu sudut pandang bisa goyah jika foto agak buram, gelap, atau miring. Rata-rata dari beberapa versi membuat prediksi **lebih stabil** dan *confidence* lebih jujur, sehingga juga membantu confidence threshold dari Tugas-1.

**Bukti dampak:** dibandingkan tanpa vs dengan TTA pada (1) data test bersih, (2) data test yang diberi gangguan (gelap, terang, blur, noise, rotasi 90°), dan (3) input di luar domain.

In [ ]:
def tta_views(X):
    return [X, X[:, :, ::-1], X[:, ::-1, :], X[:, ::-1, ::-1]]   # asli, flip-h, flip-v, rotasi 180

def predict_plain(X):
    return model.predict(X.astype('float32'), batch_size=BATCH_SIZE, verbose=0)

def predict_tta(X):
    return np.mean([predict_plain(np.ascontiguousarray(v)) for v in tta_views(X)], axis=0)

p_tta = predict_tta(X_test)
RESULTS['tta_clean'] = {'tanpa_tta': summarize(y_test, p_base), 'dengan_tta': summarize(y_test, p_tta)}
print(pd.DataFrame(RESULTS['tta_clean']).T.round(4).to_string())

In [ ]:
def corrupt(X, kind):
    Xf = X.astype('float32')
    if kind == 'Asli (bersih)':
        out = Xf
    elif kind == 'Gelap (-40%)':
        out = Xf * 0.6
    elif kind == 'Terang (+40%)':
        out = Xf * 1.4
    elif kind == 'Blur':
        out = tf.nn.avg_pool2d(Xf, ksize=5, strides=1, padding='SAME').numpy()
    elif kind == 'Noise Gaussian':
        out = Xf + np.random.default_rng(SEED).normal(0, 25, Xf.shape)
    elif kind == 'Rotasi 90°':
        out = np.rot90(Xf, k=1, axes=(1, 2))
    return np.clip(out, 0, 255).astype('uint8')

KINDS = ['Asli (bersih)', 'Gelap (-40%)', 'Terang (+40%)', 'Blur', 'Noise Gaussian', 'Rotasi 90°']
rob = []
examples = {}
for kind in KINDS:
    Xc = np.ascontiguousarray(corrupt(X_test, kind))
    examples[kind] = Xc[0]
    pa, pb = predict_plain(Xc), predict_tta(Xc)
    sa, sb = summarize(y_test, pa), summarize(y_test, pb)
    rob.append({'Gangguan': kind, 'Akurasi tanpa TTA': sa['accuracy'], 'Akurasi dengan TTA': sb['accuracy'],
                'Kesalahan tanpa TTA': sa['errors'], 'Kesalahan dengan TTA': sb['errors'],
                'F1 tanpa TTA': sa['f1_macro'], 'F1 dengan TTA': sb['f1_macro']})
    print(f"{kind:16s} akurasi {sa['accuracy']:.4f} -> {sb['accuracy']:.4f} | kesalahan {sa['errors']} -> {sb['errors']}")
rob = pd.DataFrame(rob).set_index('Gangguan')
rob['Selisih akurasi (poin %)'] = (rob['Akurasi dengan TTA'] - rob['Akurasi tanpa TTA']) * 100
rob.to_csv(os.path.join(RES_DIR, 'tta_robustness.csv'))
RESULTS['tta_robustness'] = rob.reset_index().to_dict(orient='records')
print('\nRata-rata akurasi pada data bergangguan (tanpa data bersih):')
noisy = rob.drop(index='Asli (bersih)')
print(f"  tanpa TTA: {noisy['Akurasi tanpa TTA'].mean():.4f} | dengan TTA: {noisy['Akurasi dengan TTA'].mean():.4f}")
RESULTS['tta_robustness_mean'] = {'tanpa_tta': float(noisy['Akurasi tanpa TTA'].mean()),
                                  'dengan_tta': float(noisy['Akurasi dengan TTA'].mean())}

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
xpos = np.arange(len(rob))
ax.bar(xpos - 0.2, rob['Akurasi tanpa TTA'] * 100, 0.4, label='Tanpa TTA', color='#8C8C8C')
ax.bar(xpos + 0.2, rob['Akurasi dengan TTA'] * 100, 0.4, label='Dengan TTA', color='#4C72B0')
for i, (a, b) in enumerate(zip(rob['Akurasi tanpa TTA'] * 100, rob['Akurasi dengan TTA'] * 100)):
    ax.text(i - 0.2, a, f'{a:.1f}', ha='center', va='bottom', fontsize=8)
    ax.text(i + 0.2, b, f'{b:.1f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(xpos); ax.set_xticklabels(rob.index, rotation=15)
ax.set_ylim(max(0, rob[['Akurasi tanpa TTA', 'Akurasi dengan TTA']].values.min() * 100 - 5), 100.5)
ax.set_ylabel('Akurasi (%)'); ax.set_title('Akurasi data test dengan berbagai gangguan: tanpa vs dengan TTA')
ax.legend(loc='lower left')
plt.tight_layout(); savefig('p1_02_tta_robustness.png'); plt.show()

fig, axes = plt.subplots(1, len(KINDS), figsize=(len(KINDS) * 2.6, 2.9))
for a, kind in zip(axes, KINDS):
    a.imshow(examples[kind]); a.set_title(kind, fontsize=9); a.axis('off')
plt.suptitle('Contoh gangguan yang diberikan pada data test')
plt.tight_layout(); savefig('p1_03_contoh_gangguan.png'); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.9))
for a, v, t in zip(axes, tta_views(X_test[:1]), ['Asli', 'Flip horizontal', 'Flip vertikal', 'Rotasi 180°']):
    a.imshow(v[0]); a.set_title(t, fontsize=9); a.axis('off')
plt.suptitle('Empat versi gambar yang dirata-ratakan pada TTA')
plt.tight_layout(); savefig('p1_04_contoh_tta.png'); plt.show()

### 3.1 TTA bersama confidence threshold (fitur Tugas-1) dan input di luar domain

In [ ]:
def synthetic_ood(n_each=40):
    r = np.random.default_rng(SEED)
    imgs, kinds = [], []
    for _ in range(n_each):
        imgs.append(r.integers(0, 256, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)); kinds.append('Noise acak')
        small = r.integers(0, 256, (8, 8, 3), dtype=np.uint8)
        imgs.append(np.asarray(Image.fromarray(small).resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))); kinds.append('Bercak warna (blur noise)')
        imgs.append(np.full((IMG_SIZE, IMG_SIZE, 3), r.integers(0, 256, 3), dtype=np.uint8)); kinds.append('Warna polos')
        c1, c2 = r.integers(0, 256, 3), r.integers(0, 256, 3)
        g = np.linspace(0, 1, IMG_SIZE)[None, :, None]
        imgs.append(np.repeat((c1 * (1 - g) + c2 * g), IMG_SIZE, axis=0).astype(np.uint8)); kinds.append('Gradasi')
        s = int(r.integers(8, 56))
        yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
        mask = ((yy // s + xx // s) % 2).astype(bool)
        pat = np.empty((IMG_SIZE, IMG_SIZE, 3), np.uint8); pat[mask] = r.integers(0, 256, 3); pat[~mask] = r.integers(0, 256, 3)
        imgs.append(pat); kinds.append('Pola kotak-kotak')
    return np.stack(imgs), kinds

X_ood, ood_kinds = synthetic_ood()
po_base, po_tta = predict_plain(X_ood), predict_tta(X_ood)

def selective(y, probs, thr):
    conf, ok = probs.max(1), probs.argmax(1) == y
    acc = conf >= thr
    return {'coverage': float(acc.mean()), 'akurasi_diterima': float(ok[acc].mean()),
            'kesalahan_ditolak': int((~ok & ~acc).sum()), 'total_kesalahan': int((~ok).sum())}

rows = []
for name, pt, po in [('Tanpa TTA', p_base, po_base), ('Dengan TTA', p_tta, po_tta)]:
    sel = selective(y_test, pt, THRESHOLD)
    rej = float((po.max(1) < THRESHOLD).mean())
    auroc = roc_auc_score(np.r_[np.ones(len(pt)), np.zeros(len(po))], np.r_[pt.max(1), po.max(1)])
    rows.append({'Skenario': name, 'Coverage test': sel['coverage'], 'Akurasi prediksi diterima': sel['akurasi_diterima'],
                 'Kesalahan test ditolak': f"{sel['kesalahan_ditolak']}/{sel['total_kesalahan']}",
                 'Input luar domain ditolak': rej, 'AUROC': auroc})
thr_tbl = pd.DataFrame(rows).set_index('Skenario')
print(f'Threshold = {THRESHOLD}')
print(thr_tbl.round(4).to_string())
thr_tbl.to_csv(os.path.join(RES_DIR, 'tta_threshold_ood.csv'))
RESULTS['tta_threshold_ood'] = thr_tbl.reset_index().to_dict(orient='records')

per_kind = pd.DataFrame({'jenis': ood_kinds, 'Tanpa TTA': po_base.max(1) < THRESHOLD,
                         'Dengan TTA': po_tta.max(1) < THRESHOLD}).groupby('jenis').mean() * 100
print('\nPersentase input luar domain yang ditolak per jenis:')
print(per_kind.round(1).to_string())
RESULTS['tta_ood_per_kind'] = per_kind.reset_index().to_dict(orient='records')

fig, ax = plt.subplots(figsize=(9, 4))
per_kind.plot(kind='barh', ax=ax, color=['#8C8C8C', '#4C72B0'])
ax.set_xlabel('% ditolak (confidence < threshold)'); ax.set_xlim(0, 105); ax.set_ylabel('')
ax.set_title(f'Penolakan input di luar domain: tanpa vs dengan TTA (threshold {THRESHOLD})')
plt.tight_layout(); savefig('p1_05_tta_ood.png'); plt.show()

In [ ]:
lat_tta = latency_ms(lambda x: predict_tta(x[None]), X_lat[:30], n=min(30, len(X_lat)))
lat_plain = latency_ms(lambda x: predict_plain(x[None]), X_lat[:30], n=min(30, len(X_lat)))
print(f'Latensi tanpa TTA: {lat_plain:.1f} ms | dengan TTA: {lat_tta:.1f} ms (4 kali prediksi)')
RESULTS['tta_latency_ms'] = {'tanpa_tta': lat_plain, 'dengan_tta': lat_tta}

## 4. Ringkasan dan penyimpanan hasil

In [ ]:
best = tfl.drop(index='Keras (.keras, Tugas-1)')
print('RINGKASAN PROYEK-1')
print(f"Fitur 3 (TFLite): ukuran {size_keras:.1f} MB -> {best.loc['TFLite int8', 'Ukuran (MB)']:.1f} MB "
      f"({best.loc['TFLite int8', 'Pengecilan ukuran']:.1f}x lebih kecil), akurasi int8 {best.loc['TFLite int8', 'accuracy']:.4f} "
      f"vs Keras {RESULTS['baseline_keras']['accuracy']:.4f}")
print(f"Fitur 4 (TTA): akurasi bersih {RESULTS['tta_clean']['tanpa_tta']['accuracy']:.4f} -> {RESULTS['tta_clean']['dengan_tta']['accuracy']:.4f}; "
      f"rata-rata akurasi data bergangguan {RESULTS['tta_robustness_mean']['tanpa_tta']:.4f} -> {RESULTS['tta_robustness_mean']['dengan_tta']:.4f}")

RESULTS['environment'] = {'Python': platform.python_version(), 'TensorFlow': tf.__version__, 'Keras': keras.__version__,
                          'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
                          'Matplotlib': matplotlib.__version__, 'Seaborn': sns.__version__, 'Pillow': Image.__version__}
def to_builtin(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return str(o)
with open(os.path.join(RES_DIR, 'metrics_proyek1.json'), 'w') as f:
    json.dump(RESULTS, f, indent=2, default=to_builtin)
for k, v in RESULTS['environment'].items():
    print(f'{k:13s}: {v}')

zip_path = os.path.join(os.path.dirname(OUT_DIR), 'hasil_proyek1.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(os.path.join(OUT_DIR, '**', '*'), recursive=True):
        if os.path.isfile(p):
            z.write(p, os.path.relpath(p, os.path.dirname(OUT_DIR)))
print('\nSemua hasil dikemas di:', zip_path, '(%.1f MB)' % (os.path.getsize(zip_path) / 1e6))